# Web Crawling Berita Detik

## Pendahuluan

Web crawling merupakan proses pengambilan data dari halaman web secara otomatis menggunakan program.

Pada praktikum ini, proses web crawling dilakukan menggunakan bahasa pemrograman Python untuk mengambil data berita dari situs Detik. Data yang dikumpulkan berasal dari dua kategori berita, yaitu **Sport** dan **Finance**.

Sumber data yang digunakan adalah:

- Sport: `https://sport.detik.com/`
- Finance: `https://finance.detik.com/`

Setiap berita yang berhasil dikumpulkan akan disimpan sebagai dataset dengan tiga atribut, yaitu:

- `id` — nomor identitas data
- `isi_berita` — isi atau teks berita
- `label` — kategori berita

Target pengumpulan data adalah **100 berita Sport dan 100 berita Finance**, sehingga diharapkan diperoleh maksimal **200 data berita**.

Dataset hasil crawling selanjutnya dapat digunakan untuk proses **text preprocessing, text mining, klasifikasi teks, dan machine learning**.

## 1. Import Library

Tahap pertama adalah mengimpor library yang dibutuhkan untuk proses web crawling.

Library yang digunakan terdiri dari:

- **Requests** untuk mengirim HTTP request dan mengambil halaman web.
- **BeautifulSoup** untuk membaca dan memproses struktur HTML.
- **CSV** untuk menyimpan hasil crawling ke dalam file CSV.
- **Time** untuk memberikan jeda antar-request.
- **URLJoin** untuk mengubah URL relatif menjadi URL lengkap.

Pada tahap ini program hanya mempersiapkan library dan belum melakukan proses crawling.

In [2]:
%pip install requests beautifulsoup4

  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached charset_normalizer-3.5.1-cp313-cp313-win_amd64.whl.metadata (46 kB)
  Using cached urllib3-2.7.0-py3-none-any.whl.metadata (6.9 kB)
  Using cached certifi-2026.7.22-py3-none-any.whl.metadata (2.5 kB)
Using cached requests-2.34.2-py3-none-any.whl (73 kB)
Using cached certifi-2026.7.22-py3-none-any.whl (136 kB)
Using cached charset_normalizer-3.5.1-cp313-cp313-win_amd64.whl (199 kB)
Using cached urllib3-2.7.0-py3-none-any.whl (131 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import requests
from bs4 import BeautifulSoup
import csv
import time
from urllib.parse import urljoin

## 2. Menentukan HTTP Headers

HTTP headers digunakan untuk memberikan informasi tambahan ketika program melakukan request ke sebuah website.

Pada tahap ini digunakan `User-Agent` yang menyerupai browser. Hal tersebut membuat request memiliki informasi mengenai browser yang digunakan ketika mengakses halaman website.

Header ini nantinya akan digunakan pada setiap request selama proses crawling.

In [4]:
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/131.0.0.0 Safari/537.36"
    )
}

## 3. Membuat Session

`requests.Session()` digunakan untuk membuat session yang dapat digunakan berulang kali selama proses crawling.

Dengan menggunakan session, konfigurasi seperti `User-Agent` dapat diterapkan pada setiap request secara otomatis. Dengan demikian, kita tidak perlu menuliskan konfigurasi header berulang kali.

In [5]:
session = requests.Session()
session.headers.update(HEADERS)

## 4. Membuat Fungsi untuk Mengambil URL Artikel

Tahap berikutnya adalah membuat fungsi `ambil_url_artikel()` untuk mencari URL artikel dari halaman kategori berita.

Fungsi ini akan:

1. Membuka halaman kategori berita.
2. Membuka halaman indeks berikutnya jika jumlah artikel belum terpenuhi.
3. Membaca struktur HTML menggunakan BeautifulSoup.
4. Mencari seluruh link pada halaman.
5. Mengambil URL dari atribut `href`.
6. Mengubah URL relatif menjadi URL lengkap.
7. Memastikan URL berasal dari domain yang sesuai.
8. Memastikan URL merupakan URL artikel.
9. Menghindari URL yang sama atau duplikat.
10. Menghentikan proses jika jumlah URL yang dibutuhkan sudah terpenuhi.

Fungsi ini belum akan dijalankan sampai kita memanggilnya pada tahap berikutnya.

In [6]:
def ambil_url_artikel(url_kategori, domain, jumlah=100):
    """
    Mengambil URL artikel dari homepage dan
    halaman indeks sampai jumlah URL terpenuhi.
    """

    print(f"\nMengambil artikel dari: {url_kategori}")

    daftar_url = []
    halaman = 1

    while len(daftar_url) < jumlah:

        if halaman == 1:
            url = url_kategori
        else:
            url = f"{url_kategori.rstrip('/')}/indeks?page={halaman}"

        print(f"\nMengambil halaman {halaman}:")
        print(url)

        try:
            response = session.get(
                url,
                timeout=15
            )

            print("Status:", response.status_code)

        except requests.RequestException as error:
            print("Error:", error)
            halaman += 1
            continue

        if response.status_code != 200:
            print("Halaman gagal diakses.")
            halaman += 1
            continue

        soup = BeautifulSoup(
            response.text,
            "html.parser"
        )

        jumlah_sebelum = len(daftar_url)

        for link in soup.find_all("a", href=True):

            href = link["href"].strip()

            href = urljoin(url, href)

            if (
                domain in href
                and "/d-" in href
            ):

                href = href.split("#")[0]

                if href not in daftar_url:

                    daftar_url.append(href)

                    print(
                        f"  [{len(daftar_url)}/{jumlah}] "
                        f"{href}"
                    )

                if len(daftar_url) >= jumlah:
                    break

        jumlah_baru = len(daftar_url) - jumlah_sebelum

        print(
            f"Artikel baru dari halaman ini: "
            f"{jumlah_baru}"
        )

        if jumlah_baru == 0:
            print(
                "Tidak ada artikel baru di halaman ini."
            )

        halaman += 1

        time.sleep(1)

        if halaman > 30:
            print("Batas halaman tercapai.")
            break

    print("\n======================================")
    print(
        f"Total URL {domain}: "
        f"{len(daftar_url)}"
    )
    print("======================================")

    return daftar_url[:jumlah]

## 5. Menentukan URL Kategori Sport

Sebelum menjalankan fungsi crawling, kita menentukan alamat halaman kategori Sport yang akan digunakan sebagai sumber data.

Pada tahap ini belum dilakukan pengambilan 100 berita. Kita akan melakukan pengujian terlebih dahulu dengan jumlah data yang kecil untuk memastikan fungsi dapat berjalan dengan baik.

In [7]:
sport_url = "https://sport.detik.com/"

## 6. Pengujian Pengambilan URL Artikel

Sebelum melakukan crawling dalam jumlah besar, kita melakukan pengujian terlebih dahulu.

Pada pengujian ini hanya akan diambil **5 URL artikel** dari kategori Sport.

Pengujian ini bertujuan untuk memastikan bahwa program dapat mengakses halaman kategori dan menemukan URL artikel dengan benar.

Jika proses pengujian berhasil, jumlah data nantinya dapat ditingkatkan menjadi 100 artikel.

In [8]:
url_test = ambil_url_artikel(
    sport_url,
    "sport.detik.com",
    5
)


Mengambil artikel dari: https://sport.detik.com/

Mengambil halaman 1:
https://sport.detik.com/
Status: 200
  [1/5] https://sport.detik.com/raket/d-8653585/asian-games-2026-putri-kw-pede-dengan-komposisi-beregu-putri
  [2/5] https://sport.detik.com/basket/d-8649812/derrick-michael-wujudkan-mimpi-masa-kecil-tampil-di-asian-games-2026
  [3/5] https://sport.detik.com/moto-gp/d-8652425/dokter-perkirakan-kondisi-lengan-marc-marquez-sekitar-50-persen
  [4/5] https://sport.detik.com/raket/d-8653493/tekad-alwi-farhan-lampaui-batas-di-asian-games-2026
  [5/5] https://sport.detik.com/moto-gp/d-8650852/jadwal-motogp-san-marino-2026-misi-marc-marquez-kejar-jorge-martin
Artikel baru dari halaman ini: 5

Total URL sport.detik.com: 5


## 7. Membuat Fungsi untuk Mengambil Isi Berita

Setelah URL artikel berhasil diperoleh, tahap selanjutnya adalah mengambil isi dari setiap halaman artikel.

Fungsi `ambil_isi_berita()` digunakan untuk membuka halaman artikel dan mencari bagian HTML yang berisi isi berita.

Program menggunakan beberapa selector sebagai alternatif untuk menemukan bagian artikel.

Setelah bagian artikel ditemukan, beberapa elemen yang tidak diperlukan akan dihapus, seperti:

- `script`
- `style`
- `iframe`
- `img`
- `video`
- `figure`
- `button`

Setelah itu, teks berita diambil dan dirapikan sehingga spasi yang berlebihan dihilangkan.

Hasil akhirnya berupa teks berita yang dapat digunakan sebagai data.

In [9]:
def ambil_isi_berita(url):
    """
    Membuka halaman artikel dan mengambil isi berita.
    """

    try:

        response = session.get(
            url,
            timeout=15
        )

        if response.status_code != 200:

            print(
                "Gagal membuka:",
                response.status_code
            )

            return ""

        soup = BeautifulSoup(
            response.text,
            "html.parser"
        )

        selector_list = [
            ".detail__body-text",
            ".detail__body",
            "div.detail__body-text",
            "article"
        ]

        artikel = None

        for selector in selector_list:

            artikel = soup.select_one(selector)

            if artikel is not None:
                break

        if artikel is None:
            return ""

        for tag in artikel.find_all(
            [
                "script",
                "style",
                "iframe",
                "img",
                "video",
                "figure",
                "button"
            ]
        ):

            tag.decompose()

        isi = artikel.get_text(
            " ",
            strip=True
        )

        isi = " ".join(
            isi.split()
        )

        return isi

    except requests.RequestException as error:

        print(
            "Error:",
            error
        )

        return ""

## 8. Pengujian Pengambilan Isi Berita

Selanjutnya kita melakukan pengujian terhadap satu URL artikel yang sebelumnya telah ditemukan.

Tujuannya adalah memastikan bahwa fungsi `ambil_isi_berita()` berhasil menemukan dan mengambil teks dari halaman artikel.

Jika isi berita berhasil ditampilkan, fungsi tersebut dapat digunakan untuk memproses banyak artikel.

In [10]:
url_contoh = url_test[0]

isi_contoh = ambil_isi_berita(url_contoh)

print(isi_contoh)

Jakarta - Putri Kusuma Wardani percaya diri dengan komposisi beregu putri yang dipilih PBSI untuk Asian Games 2026. Dia juga menyebut chemistry satu sama lain sudah klop. Bulutangkis Indonesia mengirimkan 20 wakil untuk tampil di Aichi-Nagoya, 19 September hingga 4 Oktober mendatang. 10 Atlet di antaranya merupakan sektor beregu putri. Yaitu Putri KW, Thalita R. Wiryawan, Ni Kadek Dhinda, Mutiara Ayu Puspitasari, Rachel Allessya Rose, Febi Setianingrum, Febriana Dwipuji Kusuma, Meilysa Trias Puspitasari, Siti Fadia Silva Ramadhanti, dan Nita Violina Marwah. SCROLL TO CONTINUE WITH CONTENT Baca juga: Tekad Alwi Farhan Lampaui Batas di Asian Games 2026 Putri menilai komposisi skuad bulutangkis Indonesia yang mengombinasikan pemain senior dan junior, tidak menemukan kendala dalam membangun koneksi di dalam tim. ADVERTISEMENT Pengalaman berlaga bersama dalam beberapa kejuaraan beregu sebelumnya dinilai telah membentuk kekompakan antar-atlet. "Dengan tim komposisi yang ada cukup kuat sih. K

## 9. Membuat Fungsi Crawling Kategori

Setelah fungsi untuk mengambil URL dan fungsi untuk mengambil isi berita berhasil dibuat dan diuji, kedua fungsi tersebut digunakan dalam satu fungsi utama, yaitu `crawling_kategori()`.

Fungsi ini bertugas melakukan crawling pada satu kategori berita.

Prosesnya adalah:

1. Mengambil URL artikel.
2. Membuka setiap URL artikel.
3. Mengambil isi berita.
4. Menyimpan isi berita.
5. Memberikan label sesuai kategori.
6. Memberikan jeda antar-request.
7. Menghentikan proses ketika jumlah data valid telah mencapai target.

Setiap data akan memiliki struktur:

```text
isi_berita → isi teks berita
label      → kategori berita

In [13]:

# 19. Code Cell — Fungsi Crawling

def crawling_kategori(
    url_kategori,
    domain,
    label,
    jumlah=100
):

    print("\n======================================")
    print(f"CRAWLING {label.upper()}")
    print("Target:", jumlah, "berita")
    print("======================================")

    urls = ambil_url_artikel(
        url_kategori,
        domain,
        jumlah
    )

    print(
        f"\nURL yang akan diproses: "
        f"{len(urls)}"
    )

    data = []

    for i, url in enumerate(
        urls,
        start=1
    ):

        print(
            f"\n[{i}/{len(urls)}] "
            f"Mengambil berita {label}..."
        )

        isi = ambil_isi_berita(url)

        if isi:

            data.append({
                "isi_berita": isi,
                "label": label
            })

            print(
                "✓ Berhasil | "
                f"{len(isi)} karakter"
            )

        else:

            print(
                "✗ Isi berita tidak ditemukan"
            )

        time.sleep(1)

        if len(data) >= jumlah:
            break

    print("\n======================================")
    print(
        f"Berita {label} berhasil: "
        f"{len(data)}"
    )
    print("======================================")

    return data

## 10. Menentukan Sumber Data

Pada tahap ini kita menentukan dua kategori berita yang akan digunakan sebagai sumber dataset.

Kategori pertama adalah **Sport**, sedangkan kategori kedua adalah **Finance**.

Kedua kategori akan diproses secara terpisah agar setiap berita dapat diberikan label yang sesuai.

In [14]:
sport_url = "https://sport.detik.com/"
finance_url = "https://finance.detik.com/"

## 11. Crawling Berita Sport

Pada tahap ini kita mulai melakukan crawling terhadap kategori **Sport**.

Program akan mengambil URL artikel dan kemudian mengambil isi dari setiap artikel.

Untuk pengujian awal, kita menggunakan target **5 berita** terlebih dahulu.

Jika proses berjalan dengan baik, target dapat diubah menjadi **100 berita**.

In [20]:
data_sport = crawling_kategori(
    sport_url,
    "sport.detik.com",
    "sport",
    5
)


CRAWLING SPORT
Target: 5 berita

Mengambil artikel dari: https://sport.detik.com/

Mengambil halaman 1:
https://sport.detik.com/
Status: 200
  [1/5] https://sport.detik.com/raket/d-8653585/asian-games-2026-putri-kw-pede-dengan-komposisi-beregu-putri
  [2/5] https://sport.detik.com/basket/d-8649812/derrick-michael-wujudkan-mimpi-masa-kecil-tampil-di-asian-games-2026
  [3/5] https://sport.detik.com/moto-gp/d-8652425/dokter-perkirakan-kondisi-lengan-marc-marquez-sekitar-50-persen
  [4/5] https://sport.detik.com/raket/d-8653493/tekad-alwi-farhan-lampaui-batas-di-asian-games-2026
  [5/5] https://sport.detik.com/moto-gp/d-8650852/jadwal-motogp-san-marino-2026-misi-marc-marquez-kejar-jorge-martin
Artikel baru dari halaman ini: 5

Total URL sport.detik.com: 5

URL yang akan diproses: 5

[1/5] Mengambil berita sport...
✓ Berhasil | 2720 karakter

[2/5] Mengambil berita sport...
✓ Berhasil | 2749 karakter

[3/5] Mengambil berita sport...
✓ Berhasil | 2191 karakter

[4/5] Mengambil berita sport.

In [21]:
data_finance = crawling_kategori(
    finance_url,
    "finance.detik.com",
    "finance",
    5
)


CRAWLING FINANCE
Target: 5 berita

Mengambil artikel dari: https://finance.detik.com/

Mengambil halaman 1:
https://finance.detik.com/
Status: 200
  [1/5] https://finance.detik.com/berita-ekonomi-bisnis/d-8653106/bandara-soetta-husein-dibuka-lagi-radin-inten-masih-ditutup
  [2/5] https://finance.detik.com/berita-ekonomi-bisnis/d-8652990/pramono-mau-terbitkan-obligasi-purbaya-kalau-uangnya-banyak-buat-apa-utang
  [3/5] https://finance.detik.com/moneter/d-8652973/muncul-isu-gaji-asn-dipindah-ke-bank-bumn-purbaya-buka-suara
  [4/5] https://finance.detik.com/energi/d-8652949/terindikasi-isi-bbm-berulang-5-000-nomor-polisi-kendaraan-diajukan-blokir
  [5/5] https://finance.detik.com/berita-ekonomi-bisnis/d-8652862/7-bandara-ditutup-hingga-23-59-wib-ini-daftarnya
Artikel baru dari halaman ini: 5

Total URL finance.detik.com: 5

URL yang akan diproses: 5

[1/5] Mengambil berita finance...
✓ Berhasil | 2459 karakter

[2/5] Mengambil berita finance...
✓ Berhasil | 2758 karakter

[3/5] Mengambil

In [23]:
data_sport = crawling_kategori(
    sport_url,
    "sport.detik.com",
    "sport",
    100
)


CRAWLING SPORT
Target: 100 berita

Mengambil artikel dari: https://sport.detik.com/

Mengambil halaman 1:
https://sport.detik.com/
Status: 200
  [1/100] https://sport.detik.com/raket/d-8653585/asian-games-2026-putri-kw-pede-dengan-komposisi-beregu-putri
  [2/100] https://sport.detik.com/basket/d-8649812/derrick-michael-wujudkan-mimpi-masa-kecil-tampil-di-asian-games-2026
  [3/100] https://sport.detik.com/moto-gp/d-8652425/dokter-perkirakan-kondisi-lengan-marc-marquez-sekitar-50-persen
  [4/100] https://sport.detik.com/raket/d-8653493/tekad-alwi-farhan-lampaui-batas-di-asian-games-2026
  [5/100] https://sport.detik.com/moto-gp/d-8650852/jadwal-motogp-san-marino-2026-misi-marc-marquez-kejar-jorge-martin
  [6/100] https://sport.detik.com/sport-lain/d-8654342/menuju-asian-games-2026-ketum-koi-tekankan-kolaborasi
  [7/100] https://sport.detik.com/sport-lain/d-8649149/timnas-voli-putra-dan-putri-indonesia-bidik-8-besar-asian-games-2026
  [8/100] https://sport.detik.com/sport-lain/d-8648160/

In [24]:
data_finance = crawling_kategori(
    finance_url,
    "finance.detik.com",
    "finance",
    100
)


CRAWLING FINANCE
Target: 100 berita

Mengambil artikel dari: https://finance.detik.com/

Mengambil halaman 1:
https://finance.detik.com/
Status: 200
  [1/100] https://finance.detik.com/berita-ekonomi-bisnis/d-8653106/bandara-soetta-husein-dibuka-lagi-radin-inten-masih-ditutup
  [2/100] https://finance.detik.com/berita-ekonomi-bisnis/d-8652990/pramono-mau-terbitkan-obligasi-purbaya-kalau-uangnya-banyak-buat-apa-utang
  [3/100] https://finance.detik.com/moneter/d-8652973/muncul-isu-gaji-asn-dipindah-ke-bank-bumn-purbaya-buka-suara
  [4/100] https://finance.detik.com/energi/d-8652949/terindikasi-isi-bbm-berulang-5-000-nomor-polisi-kendaraan-diajukan-blokir
  [5/100] https://finance.detik.com/berita-ekonomi-bisnis/d-8652862/7-bandara-ditutup-hingga-23-59-wib-ini-daftarnya
  [6/100] https://finance.detik.com/infrastruktur/d-8654568/513-9-km-listrik-aliran-atas-krl-jabodetabek-dirawat-ini-prosesnya
  [7/100] https://finance.detik.com/infrastruktur/d-8640936/154-gardu-perlintasan-sebidang-di

## 13. Menggabungkan Data

Setelah proses crawling kategori Sport dan Finance selesai, kedua dataset digabungkan menjadi satu variabel bernama `semua_data`.

Penggabungan dilakukan agar seluruh berita dapat disimpan dalam satu file dataset.

Jika masing-masing kategori berhasil mendapatkan 100 berita, maka jumlah keseluruhan data adalah:

- Sport = 100 berita
- Finance = 100 berita
- Total = 200 berita

In [25]:
semua_data = (
    data_sport +
    data_finance
)

print("Total data:", len(semua_data))

Total data: 200


## 14. Menyimpan Dataset ke CSV

Data hasil crawling selanjutnya disimpan ke dalam file CSV dengan nama `dataset_berita.csv`.

Format CSV dipilih karena mudah digunakan dan dapat dibuka menggunakan berbagai aplikasi seperti Microsoft Excel, Google Sheets, maupun Python.

Dataset terdiri dari tiga kolom:

| Kolom | Keterangan |
|---|---|
| `id` | Nomor identitas data |
| `isi_berita` | Teks atau isi berita |
| `label` | Kategori berita |

Kolom `id` dibuat secara berurutan mulai dari angka 1.

In [26]:
with open(
    "dataset_berita.csv",
    "w",
    newline="",
    encoding="utf-8-sig"
) as file:

    writer = csv.writer(file)

    writer.writerow([
        "id",
        "isi_berita",
        "label"
    ])

    for i, berita in enumerate(
        semua_data,
        start=1
    ):

        writer.writerow([
            i,
            berita["isi_berita"],
            berita["label"]
        ])

print("Dataset berhasil disimpan sebagai dataset_berita.csv")

Dataset berhasil disimpan sebagai dataset_berita.csv


## 15. Hasil Crawling

Tahap terakhir adalah menampilkan ringkasan hasil proses crawling.

Informasi yang ditampilkan meliputi jumlah berita yang berhasil dikumpulkan dari setiap kategori, jumlah keseluruhan data, serta nama file dataset yang telah dibuat.

Apabila seluruh target berhasil dicapai, maka jumlah data yang dihasilkan adalah:

```text
Sport    : 100 berita
Finance  : 100 berita
Total    : 200 berita

In [27]:

# 31. Code Cell — Hasil Akhir

print("======================================")
print("        CRAWLING SELESAI")
print("======================================")

print("Data Sport   :", len(data_sport))
print("Data Finance :", len(data_finance))
print("Total Data   :", len(semua_data))
print("File         : dataset_berita.csv")

print("======================================")

        CRAWLING SELESAI
Data Sport   : 100
Data Finance : 100
Total Data   : 200
File         : dataset_berita.csv


## 16. Menampilkan Data Hasil Crawling

Setelah proses crawling selesai dan dataset berhasil disimpan, kita dapat melihat sebagian data yang telah berhasil dikumpulkan.

Pada tahap ini, data ditampilkan berdasarkan kategorinya, yaitu **Sport** dan **Finance**.

Masing-masing kategori akan menampilkan 5 data pertama sebagai contoh hasil crawling. Data yang ditampilkan terdiri dari:

* `id` — nomor identitas data
* `isi_berita` — isi berita
* `label` — kategori berita

Penampilan sebagian data ini bertujuan untuk memastikan bahwa proses crawling dan penyimpanan dataset telah berjalan dengan baik.


In [28]:
%pip install pandas

   ---------------------------------------- 0.0/9.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.8 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.8 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.8 MB ? eta -:--:--
   -- ------------------------------------- 0.5/9.8 MB 654.4 kB/s eta 0:00:15
   --- ------------------------------------ 0.8/9.8 MB 891.9 kB/s eta 0:00:11
   --- ------------------------------------ 0.8/9.8 MB 891.9 kB/s eta 0:00:11
   ---- ----------------------------------- 1.0/9.8 MB 730.4 kB/s eta 0:00:13
   ---- ----------------------------------- 1.0/9.8 MB 730.4 kB/s eta 0:00:13
   ---- ----------------------------------- 1.0/9.8 MB 730.4 kB/s eta 0:00:13
   ---- ----------------------------------- 1.0/9.8 MB 730.4 kB/s eta 0:00:13
   ----- ---------------------------------- 1.3/9.8 MB 560.1 kB/s eta 0:00:16
   ----- ---------------------------------- 1.3/9.8 MB 560.1 kB/s eta 0:00:16
   ----- -------------


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [29]:
import pandas as pd

# Membaca dataset hasil crawling
df = pd.read_csv("dataset_berita.csv")

print("Jumlah seluruh data:", len(df))

Jumlah seluruh data: 200


In [32]:
print("===== 5 DATA SPORT =====")

display(
    df[df["label"] == "sport"][
        ["id", "isi_berita", "label"]
    ].head(5)
)

print("\n===== 5 DATA FINANCE =====")

display(
    df[df["label"] == "finance"][
        ["id", "isi_berita", "label"]
    ].head(5)
)

===== 5 DATA SPORT =====


,id,isi_berita,label
0,1,Jakarta - Putri Kusuma Wardani percaya diri de...,sport
1,2,Jakarta - Mata Derrick Michael Xzavierro berbi...,sport
2,3,Jakarta - Marc Marquez masih harus beradaptasi...,sport
3,4,Jakarta - Momen debut di Asian Games 2026 tak ...,sport
4,5,Jakarta - MotoGP 2026 akan berlanjut ke San Ma...,sport



===== 5 DATA FINANCE =====


,id,isi_berita,label
100,101,Jakarta - Penutupan operasional Bandara Soekar...,finance
101,102,Jakarta - Menteri Keuangan (Menkeu) Purbaya Yu...,finance
102,103,Jakarta - Menteri Keuangan Purbaya Yudhi Sadew...,finance
103,104,Jakarta - PT Pertamina Patra Niaga memperkuat ...,finance
104,105,Jakarta - AirNav Indonesia kembali memperbarui...,finance
